In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "rank_bm25", "-q"])

import json, re
import numpy as np
import pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi

TEST_DIR    = "./data_outputs/task2/test"
CONTEXT_MODE = "window_1"
RUN_NAME    = "bm25-baseline-window_1"

print("✅ Libraries ready")

✅ Libraries ready


In [2]:
def get_context(text, citation_id, mode="window_1"):
    if mode == "full":
        return text
    window = int(mode.split("_")[1])
    sentences = re.split(r"(?<=[.!?])\s+", text)
    target_idx = next((i for i, s in enumerate(sentences) if citation_id in s), -1)
    if target_idx == -1:
        return text
    start = max(0, target_idx - window)
    end   = min(len(sentences), target_idx + window + 1)
    return " ".join(sentences[start:end])


def load_ranking_groups(test_dir, context_mode):
    label_files = sorted(Path(test_dir).glob("*.label"))
    total = len(label_files)
    print(f"📊 Loading {total:,} test files...")

    groups, skipped = [], 0
    for idx, lf in enumerate(label_files):
        if (idx + 1) % 500 == 0:
            print(f"  ⏳ {idx+1:,}/{total:,}")
        try:
            in_data    = json.loads(lf.with_suffix(".in").read_text())
            label_data = json.loads(lf.read_text())
        except Exception:
            skipped += 1; continue

        text        = in_data.get("text", "")
        candidates  = in_data.get("citation_candidates", [])
        bib_entries = in_data.get("bib_entries", {})
        correct_map = label_data.get("correct_citation", {})

        if not text or not candidates or not bib_entries or not correct_map:
            skipped += 1; continue

        for citation_id, correct_paper_id in correct_map.items():
            context = get_context(text, citation_id, mode=context_mode)
            cands = [
                {
                    "paper_id":   pid,
                    "text":       "{} {}".format(
                        bib_entries[pid].get("title", ""),
                        bib_entries[pid].get("abstract", "")
                    ),
                    "is_correct": pid == correct_paper_id,
                }
                for pid in candidates if pid in bib_entries
            ]
            if cands:
                groups.append({"context": context, "candidates": cands})

    print(f"\n✅ {len(groups):,} ranking groups | skipped: {skipped}")
    return groups


groups = load_ranking_groups(TEST_DIR, CONTEXT_MODE)

📊 Loading 3,000 test files...
  ⏳ 500/3,000
  ⏳ 1,000/3,000
  ⏳ 1,500/3,000
  ⏳ 2,000/3,000
  ⏳ 2,500/3,000
  ⏳ 3,000/3,000

✅ 7,151 ranking groups | skipped: 0


In [3]:
def tokenize(text):
    return re.sub(r"[^\w\s]", " ", text.lower()).split()


def bm25_rank(group):
    """Return rank (1-indexed) of correct paper using BM25."""
    corpus  = [tokenize(c["text"]) for c in group["candidates"]]
    query   = tokenize(group["context"])
    bm25    = BM25Okapi(corpus)
    scores  = bm25.get_scores(query)
    ranked  = np.argsort(scores)[::-1]
    for rank, idx in enumerate(ranked):
        if group["candidates"][idx]["is_correct"]:
            return rank + 1
    return None


print("✅ BM25 ranking function ready")

✅ BM25 ranking function ready


In [4]:
print(f"Running BM25 on {len(groups):,} queries...")

reciprocal_ranks = []
hits  = {1: 0, 3: 0, 5: 0}
ranks = []

for i, g in enumerate(groups):
    if (i + 1) % 1000 == 0:
        print(f"  ⏳ {i+1:,}/{len(groups):,}")
    r = bm25_rank(g)
    ranks.append(r)
    if r is None:
        reciprocal_ranks.append(0.0)
        continue
    reciprocal_ranks.append(1.0 / r)
    for k in hits:
        if r <= k:
            hits[k] += 1

n   = len(groups)
mrr = float(np.mean(reciprocal_ranks))
print(f"\n✅ Done")

Running BM25 on 7,151 queries...
  ⏳ 1,000/7,151
  ⏳ 2,000/7,151
  ⏳ 3,000/7,151
  ⏳ 4,000/7,151
  ⏳ 5,000/7,151
  ⏳ 6,000/7,151
  ⏳ 7,000/7,151

✅ Done


In [5]:
bm25_results = {
    "Model":    "BM25 (baseline)",
    "Queries":  n,
    "MRR":      round(mrr, 4),
    "Hits@1":   round(hits[1] / n, 4),
    "Hits@3":   round(hits[3] / n, 4),
    "Hits@5":   round(hits[5] / n, 4),
}

scibert_results = {
    "Model":    "SciBERT (fine-tuned)",
    "Queries":  7151,
    "MRR":      0.5855,
    "Hits@1":   0.4159,
    "Hits@3":   0.6925,
    "Hits@5":   0.8130,
}

df = pd.DataFrame([bm25_results, scibert_results]).set_index("Model")

print("=" * 65)
print(f"📊 COMPARISON: BM25 Baseline vs SciBERT | {CONTEXT_MODE}")
print("=" * 65)
print(df.to_string())
print("=" * 65)
print()
print(f"  BM25    → MRR: {bm25_results['MRR']:.4f} | Hits@1: {bm25_results['Hits@1']:.4f} | Hits@3: {bm25_results['Hits@3']:.4f} | Hits@5: {bm25_results['Hits@5']:.4f}")
print(f"  SciBERT → MRR: {scibert_results['MRR']:.4f} | Hits@1: {scibert_results['Hits@1']:.4f} | Hits@3: {scibert_results['Hits@3']:.4f} | Hits@5: {scibert_results['Hits@5']:.4f}")
print(f"")
print(f"  Δ MRR   : +{scibert_results['MRR'] - bm25_results['MRR']:+.4f}")
print(f"  Δ Hits@1: +{scibert_results['Hits@1'] - bm25_results['Hits@1']:+.4f}")
print(f"  Δ Hits@3: +{scibert_results['Hits@3'] - bm25_results['Hits@3']:+.4f}")
print(f"  Δ Hits@5: +{scibert_results['Hits@5'] - bm25_results['Hits@5']:+.4f}")

📊 COMPARISON: BM25 Baseline vs SciBERT | window_1
                      Queries     MRR  Hits@1  Hits@3  Hits@5
Model                                                        
BM25 (baseline)          7151  0.4691  0.2923  0.5574  0.6932
SciBERT (fine-tuned)     7151  0.5855  0.4159  0.6925  0.8130

  BM25    → MRR: 0.4691 | Hits@1: 0.2923 | Hits@3: 0.5574 | Hits@5: 0.6932
  SciBERT → MRR: 0.5855 | Hits@1: 0.4159 | Hits@3: 0.6925 | Hits@5: 0.8130

  Δ MRR   : ++0.1164
  Δ Hits@1: ++0.1236
  Δ Hits@3: ++0.1351
  Δ Hits@5: ++0.1198
